# YOLOX Fine-Tuning: Add Single MSL Class (98 Total Classes)

Fine-tunes the 97-class hazmat model to add **1 new class** for Military Shipping Labels.

## Simple Goal
- **Class 97:** `militaryShippingLabel` - Detects ANY MSL variant
- App logic: "Does this package have an MSL?" → Yes/No

## What Counts as an MSL (All Use Class 97)
- Full 4x6" Military Shipping Labels
- Container ID labels (exterior, intermediate, unit)
- Priority designator circles (1, 2, 3)
- PDF417 barcodes on military shipments
- Address blocks with DODAAC codes
- Any MIL-STD-129 compliant marking

## Data Requirements
- **Minimum:** 200-300 MSL images
- **Recommended:** 500+ MSL images
- Include variety: different MSL types, lighting, angles, wear

## Prerequisites
1. Base checkpoint: `confidence_boost_epoch_60.pth`
2. MSL dataset in YOLO format (all labeled as class 97)
3. GPU runtime (T4 or better)

## Cell 1: Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Enable in Runtime > Change runtime type")

## Cell 2: Configuration

In [ ]:
import os
import json

# ============================================
# PATHS - UPDATE FOR YOUR SETUP
# ============================================
DRIVE_ROOT = "/content/drive/MyDrive"

# Base checkpoint (97-class hazmat model)
BASE_CHECKPOINT = f"{DRIVE_ROOT}/HazProML/models/confidence_boost_97class/confidence_boost_epoch_60.pth"

# Alternative paths
ALT_CHECKPOINTS = [
    f"{DRIVE_ROOT}/hazmat_models/confidence_boost/confidence_boost_final.pth",
    f"{DRIVE_ROOT}/HazProML/models/confidence_boost_97class/confidence_boost_final.pth",
]

# MSL dataset (YOUR annotated MSL images)
# All images should be labeled with class_id = 97
MSL_DATASET_DIR = f"{DRIVE_ROOT}/HazProML/data/msl_dataset"

# Existing hazmat dataset (to prevent forgetting)
HAZMAT_DATASET_DIR = f"{DRIVE_ROOT}/HazProML/data/combined_dataset"

# Output
DRIVE_OUTPUT = f"{DRIVE_ROOT}/HazProML/models/msl_98class"

# ============================================
# MODEL CONFIG
# ============================================
OLD_NUM_CLASSES = 97   # Existing hazmat
NEW_NUM_CLASSES = 98   # + 1 MSL class
MSL_CLASS_ID = 97      # The single MSL class

# YOLOX-Tiny
DEPTH = 0.33
WIDTH = 0.375
INPUT_SIZE = (640, 640)

# ============================================
# TRAINING CONFIG
# ============================================
MAX_EPOCHS = 40
BATCH_SIZE = 16
BASIC_LR = 0.00005 / 64.0  # Very low to preserve hazmat
FREEZE_BACKBONE_EPOCHS = 10
NO_AUG_EPOCHS = 15
WARMUP_EPOCHS = 3
SAVE_INTERVAL = 10

# Dataset mixing
MSL_RATIO = 0.3  # 30% MSL, 70% hazmat per batch
MAX_HAZMAT_IMAGES = 1500

# Local working dir
LOCAL_DATA_DIR = '/content/data/msl_finetune'

# ============================================
# VERIFY
# ============================================
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

ckpt_path = None
if os.path.exists(BASE_CHECKPOINT):
    ckpt_path = BASE_CHECKPOINT
else:
    for alt in ALT_CHECKPOINTS:
        if os.path.exists(alt):
            ckpt_path = alt
            break

print("=" * 60)
print("MSL FINE-TUNING CONFIG (Single Class)")
print("=" * 60)
print(f"\nCheckpoint: {ckpt_path or 'NOT FOUND'}")
print(f"MSL Dataset: {MSL_DATASET_DIR}")
print(f"  Exists: {os.path.exists(MSL_DATASET_DIR)}")
print(f"\nClasses: {OLD_NUM_CLASSES} → {NEW_NUM_CLASSES}")
print(f"New class: ID {MSL_CLASS_ID} = militaryShippingLabel")
print(f"\nTraining: {MAX_EPOCHS} epochs, LR={BASIC_LR*64:.6f}")
print(f"Output: {DRIVE_OUTPUT}")
print("=" * 60)

## Cell 3: Install YOLOX

In [ ]:
%%capture
!pip install cython pycocotools thop loguru tabulate ninja

import os
if not os.path.exists('/content/YOLOX'):
    !git clone https://github.com/Megvii-BaseDetection/YOLOX.git /content/YOLOX

%cd /content/YOLOX
!pip install -v -e .

In [ ]:
import sys
sys.path.insert(0, '/content/YOLOX')
from yolox.exp import get_exp
from yolox.models import YOLOX, YOLOPAFPN, YOLOXHead
print("YOLOX ready!")

## Cell 4: Create 98-Class Mapping

In [ ]:
import json

# Load existing 97-class mapping
hazmat_paths = [
    f"{DRIVE_ROOT}/HazProML/data/combined_dataset/class_mapping.json",
    f"{DRIVE_ROOT}/class_mapping.json",
    f"{HAZMAT_DATASET_DIR}/class_mapping.json",
]

hazmat_mapping = None
for path in hazmat_paths:
    if os.path.exists(path):
        with open(path, 'r') as f:
            hazmat_mapping = json.load(f)
        print(f"Loaded hazmat mapping: {path}")
        break

if hazmat_mapping is None:
    print("Creating placeholder hazmat mapping...")
    hazmat_mapping = {str(i): {"id": i, "name": f"hazmat_{i}", "category": "hazmat"} for i in range(97)}

# Add single MSL class
class_mapping_98 = dict(hazmat_mapping)
class_mapping_98["97"] = {
    "id": 97,
    "name": "militaryShippingLabel",
    "category": "military_msl",
    "description": "MIL-STD-129 Military Shipping Label - any variant"
}

# Save
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
local_mapping = f"{LOCAL_DATA_DIR}/class_mapping.json"
with open(local_mapping, 'w') as f:
    json.dump(class_mapping_98, f, indent=2)

drive_mapping = f"{DRIVE_OUTPUT}/class_mapping_98class.json"
with open(drive_mapping, 'w') as f:
    json.dump(class_mapping_98, f, indent=2)

print(f"\nCreated 98-class mapping:")
print(f"  Classes 0-96: Hazmat")
print(f"  Class 97: militaryShippingLabel (MSL)")
print(f"\nSaved to: {drive_mapping}")

## Cell 5: Prepare Dataset

**Your MSL annotations must use class_id = 97**

Example YOLO label file for MSL:
```
97 0.5 0.5 0.3 0.4
```

In [ ]:
import shutil
import random
from tqdm import tqdm
from pathlib import Path

# Clean
!rm -rf /content/data
os.makedirs(f'{LOCAL_DATA_DIR}/images/train', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/images/val', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{LOCAL_DATA_DIR}/labels/val', exist_ok=True)

def copy_dataset(src_img, src_lbl, dst_img, dst_lbl, prefix, max_count=None):
    if not os.path.exists(src_img):
        print(f"  Not found: {src_img}")
        return 0
    files = [f for f in os.listdir(src_img) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if max_count:
        random.shuffle(files)
        files = files[:max_count]
    for f in tqdm(files, desc=f"  {prefix}", leave=False):
        shutil.copy2(f"{src_img}/{f}", f"{dst_img}/{prefix}_{f}")
        lbl = os.path.splitext(f)[0] + '.txt'
        if os.path.exists(f"{src_lbl}/{lbl}"):
            shutil.copy2(f"{src_lbl}/{lbl}", f"{dst_lbl}/{prefix}_{lbl}")
    return len(files)

# Copy MSL
print("\nCopying MSL images...")
msl_train = copy_dataset(
    f"{MSL_DATASET_DIR}/images/train", f"{MSL_DATASET_DIR}/labels/train",
    f"{LOCAL_DATA_DIR}/images/train", f"{LOCAL_DATA_DIR}/labels/train", "msl"
)
msl_val = copy_dataset(
    f"{MSL_DATASET_DIR}/images/val", f"{MSL_DATASET_DIR}/labels/val",
    f"{LOCAL_DATA_DIR}/images/val", f"{LOCAL_DATA_DIR}/labels/val", "msl"
)
print(f"  MSL: {msl_train} train, {msl_val} val")

# Copy hazmat (balanced)
print("\nCopying hazmat images...")
target_hazmat = int(msl_train * (1 - MSL_RATIO) / MSL_RATIO) if msl_train > 0 else MAX_HAZMAT_IMAGES
target_hazmat = min(target_hazmat, MAX_HAZMAT_IMAGES)

hazmat_train = copy_dataset(
    f"{HAZMAT_DATASET_DIR}/images/train", f"{HAZMAT_DATASET_DIR}/labels/train",
    f"{LOCAL_DATA_DIR}/images/train", f"{LOCAL_DATA_DIR}/labels/train", "haz",
    max_count=target_hazmat
)
hazmat_val = copy_dataset(
    f"{HAZMAT_DATASET_DIR}/images/val", f"{HAZMAT_DATASET_DIR}/labels/val",
    f"{LOCAL_DATA_DIR}/images/val", f"{LOCAL_DATA_DIR}/labels/val", "haz",
    max_count=200
)
print(f"  Hazmat: {hazmat_train} train, {hazmat_val} val")

total_train = len(os.listdir(f"{LOCAL_DATA_DIR}/images/train"))
total_val = len(os.listdir(f"{LOCAL_DATA_DIR}/images/val"))

print(f"\n" + "=" * 50)
print(f"DATASET: {total_train} train, {total_val} val")
if msl_train > 0:
    print(f"MSL ratio: {msl_train/total_train*100:.1f}%")
else:
    print("\n⚠️  NO MSL IMAGES FOUND!")
    print(f"Upload to: {MSL_DATASET_DIR}/images/train/")
print("=" * 50)

## Cell 6: Convert to COCO Format

In [ ]:
from PIL import Image
from datetime import datetime

def yolo_to_coco(img_dir, lbl_dir, class_map, out_path, num_classes):
    categories = [{"id": int(k), "name": v["name"], "supercategory": v.get("category", "object")}
                  for k, v in class_map.items()]
    images, annotations = [], []
    ann_id = 0
    msl_count = 0

    img_files = list(Path(img_dir).glob("*.jpg")) + list(Path(img_dir).glob("*.png"))

    for img_id, img_path in enumerate(tqdm(sorted(img_files), desc="Converting")):
        try:
            with Image.open(img_path) as img:
                w, h = img.size
        except:
            continue

        images.append({"id": img_id, "file_name": img_path.name, "width": w, "height": h,
                       "license": 1, "date_captured": datetime.now().strftime("%Y-%m-%d")})

        lbl_path = Path(lbl_dir) / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue

        with open(lbl_path) as f:
            content = f.read().strip()
        if not content:
            continue

        for line in content.split('\n'):
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            if cls_id >= num_classes:
                continue
            if cls_id == 97:
                msl_count += 1

            xc, yc, bw, bh = map(float, parts[1:5])
            x = (xc - bw/2) * w
            y = (yc - bh/2) * h

            annotations.append({
                "id": ann_id, "image_id": img_id, "category_id": cls_id,
                "bbox": [round(x,2), round(y,2), round(bw*w,2), round(bh*h,2)],
                "area": round(bw*w*bh*h, 2), "iscrowd": 0, "segmentation": []
            })
            ann_id += 1

    coco = {
        "info": {"description": "Hazmat+MSL Dataset", "version": "1.0", "year": 2025},
        "licenses": [{"id": 1, "name": "MIT", "url": ""}],
        "categories": categories, "images": images, "annotations": annotations
    }
    with open(out_path, 'w') as f:
        json.dump(coco, f)

    return len(images), len(annotations), msl_count

print("Converting to COCO format...")
train_imgs, train_anns, train_msl = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/train', f'{LOCAL_DATA_DIR}/labels/train',
    class_mapping_98, f'{LOCAL_DATA_DIR}/train.json', NEW_NUM_CLASSES
)
val_imgs, val_anns, val_msl = yolo_to_coco(
    f'{LOCAL_DATA_DIR}/images/val', f'{LOCAL_DATA_DIR}/labels/val',
    class_mapping_98, f'{LOCAL_DATA_DIR}/val.json', NEW_NUM_CLASSES
)

print(f"\nTrain: {train_imgs} images, {train_anns} annotations")
print(f"  MSL annotations (class 97): {train_msl}")
print(f"  Hazmat annotations: {train_anns - train_msl}")
print(f"\nVal: {val_imgs} images, {val_anns} annotations")

## Cell 7: Create YOLOX Config

In [ ]:
exp_content = f'''#!/usr/bin/env python3
from yolox.exp import Exp as MyExp

class Exp(MyExp):
    def __init__(self):
        super().__init__()
        self.depth = {DEPTH}
        self.width = {WIDTH}
        self.num_classes = {NEW_NUM_CLASSES}
        self.act = "silu"
        self.data_dir = "{LOCAL_DATA_DIR}"
        self.train_ann = "train.json"
        self.val_ann = "val.json"
        self.data_num_workers = 4
        self.max_epoch = {MAX_EPOCHS}
        self.warmup_epochs = {WARMUP_EPOCHS}
        self.basic_lr_per_img = {BASIC_LR}
        self.scheduler = "yoloxwarmcos"
        self.min_lr_ratio = 0.01
        self.no_aug_epochs = {NO_AUG_EPOCHS}
        self.input_size = {INPUT_SIZE}
        self.test_size = {INPUT_SIZE}
        self.mosaic_prob = 0.5
        self.mixup_prob = 0.3
        self.enable_mixup = True
        self.output_dir = "/content/outputs"
        self.exp_name = "yolox_msl_98class"
        self.eval_interval = 1000
        self.save_history_ckpt = True
        self.print_interval = 50

    def get_data_loader(self, batch_size, is_distributed, no_aug=False, cache_img=None):
        from yolox.data import COCODataset, TrainTransform, YoloBatchSampler, DataLoader, InfiniteSampler, MosaicDetection, worker_init_reset_seed
        from yolox.utils import wait_for_the_master
        with wait_for_the_master():
            dataset = COCODataset(data_dir=self.data_dir, json_file=self.train_ann, img_size=self.input_size,
                preproc=TrainTransform(max_labels=50, flip_prob=0.5, hsv_prob=1.0), cache=False, name="images/train")
        dataset = MosaicDetection(dataset, mosaic=not no_aug, img_size=self.input_size,
            preproc=TrainTransform(max_labels=120, flip_prob=0.5, hsv_prob=1.0),
            degrees=5.0, translate=0.05, mosaic_scale=(0.5, 1.5), mixup_scale=(0.5, 1.5),
            shear=1.0, enable_mixup=True, mosaic_prob=0.5, mixup_prob=0.3)
        self.dataset = dataset
        sampler = InfiniteSampler(len(self.dataset), seed=self.seed if self.seed else 0)
        batch_sampler = YoloBatchSampler(sampler=sampler, batch_size=batch_size, drop_last=False, mosaic=not no_aug)
        return DataLoader(self.dataset, num_workers=self.data_num_workers, pin_memory=True,
                          batch_sampler=batch_sampler, worker_init_fn=worker_init_reset_seed)

    def get_eval_loader(self, *args, **kwargs): return None
    def get_evaluator(self, *args, **kwargs): return None
    def eval(self, model, evaluator, is_distributed, half=False, return_outputs=False):
        return (0, 0, "Disabled"), None
'''

with open('/content/YOLOX/exps/msl_98class_exp.py', 'w') as f:
    f.write(exp_content)
print("Created experiment config")

## Cell 8: Expand Model (97 → 98 classes)

In [ ]:
import torch.nn as nn

def create_model(num_classes):
    in_ch = [256, 512, 1024]
    backbone = YOLOPAFPN(DEPTH, WIDTH, in_channels=in_ch, act='silu')
    head = YOLOXHead(num_classes, WIDTH, in_channels=in_ch, act='silu')
    return YOLOX(backbone, head)

def expand_weights(old_sd, old_nc, new_nc):
    new_sd = {}
    for k, v in old_sd.items():
        if 'cls_preds' in k and v.shape[0] == old_nc:
            new_shape = (new_nc,) + v.shape[1:]
            new_v = torch.zeros(new_shape, dtype=v.dtype)
            new_v[:old_nc] = v
            if 'weight' in k:
                nn.init.xavier_uniform_(new_v[old_nc:])
            else:
                new_v[old_nc:] = -2.0  # Bias init
            new_sd[k] = new_v
            print(f"  Expanded: {k}")
        else:
            new_sd[k] = v
    return new_sd

print("=" * 60)
print("EXPANDING MODEL: 97 → 98 CLASSES")
print("=" * 60)

if not ckpt_path:
    raise FileNotFoundError("No checkpoint found!")

print(f"\nLoading: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location='cpu')
old_sd = ckpt['model'] if 'model' in ckpt else ckpt

# Detect actual class count
for k, v in old_sd.items():
    if 'cls_preds' in k and 'weight' in k:
        detected = v.shape[0]
        print(f"Detected classes: {detected}")
        if detected != OLD_NUM_CLASSES:
            OLD_NUM_CLASSES = detected
        break

print(f"\nExpanding {OLD_NUM_CLASSES} → {NEW_NUM_CLASSES}...")
new_sd = expand_weights(old_sd, OLD_NUM_CLASSES, NEW_NUM_CLASSES)

# Verify
model = create_model(NEW_NUM_CLASSES)
model.load_state_dict(new_sd, strict=False)
model.eval()

with torch.no_grad():
    out = model(torch.randn(1, 3, 640, 640))
print(f"\nOutput shape: {out.shape}")
print(f"Expected: [1, 8400, {5 + NEW_NUM_CLASSES}]")

# Save
YOLOX_OUTPUT_DIR = '/content/outputs/yolox_msl_98class'
os.makedirs(YOLOX_OUTPUT_DIR, exist_ok=True)

expanded_ckpt = {'model': new_sd, 'start_epoch': 0}
torch.save(expanded_ckpt, f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth")
print(f"\nSaved: {YOLOX_OUTPUT_DIR}/latest_ckpt.pth")

## Cell 9: Train!

In [ ]:
import threading, time, shutil

backup_running = True

def backup_loop():
    last = -1
    while backup_running:
        time.sleep(60)
        ckpt = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"
        if os.path.exists(ckpt):
            try:
                d = torch.load(ckpt, map_location='cpu')
                ep = d.get('start_epoch', 0)
                if ep > 0 and ep % SAVE_INTERVAL == 0 and ep != last:
                    shutil.copy(ckpt, f"{DRIVE_OUTPUT}/msl_98class_epoch_{ep}.pth")
                    print(f"\n*** Saved epoch {ep} ***")
                    last = ep
            except: pass

threading.Thread(target=backup_loop, daemon=True).start()

print("=" * 60)
print(f"TRAINING: 98 classes, {MAX_EPOCHS} epochs")
print(f"MSL images: {msl_train}, Hazmat images: {hazmat_train}")
print("=" * 60 + "\n")

%cd /content/YOLOX
!PYTHONPATH=/content/YOLOX python tools/train.py \
    -f exps/msl_98class_exp.py -d 1 -b {BATCH_SIZE} --fp16 -o --resume

backup_running = False
print("\n" + "=" * 60)
print("TRAINING COMPLETE!")
print("=" * 60)

## Cell 10: Save Final Model

In [ ]:
from datetime import datetime

latest = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"
if os.path.exists(latest):
    ts = datetime.now().strftime("%Y%m%d")
    shutil.copy(latest, f"{DRIVE_OUTPUT}/msl_98class_{MAX_EPOCHS}ep_{ts}.pth")
    shutil.copy(latest, f"{DRIVE_OUTPUT}/msl_98class_final.pth")
    shutil.copy(local_mapping, f"{DRIVE_OUTPUT}/class_mapping_98class.json")

    print("Saved to Google Drive:")
    for f in sorted(os.listdir(DRIVE_OUTPUT)):
        sz = os.path.getsize(f"{DRIVE_OUTPUT}/{f}") / 1024 / 1024
        print(f"  {f} ({sz:.1f} MB)")

## Cell 11: Test Inference

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from yolox.utils import postprocess
from yolox.data.data_augment import ValTransform

# Load model
final = f"{DRIVE_OUTPUT}/msl_98class_final.pth"
if not os.path.exists(final):
    final = f"{YOLOX_OUTPUT_DIR}/latest_ckpt.pth"

model = create_model(NEW_NUM_CLASSES)
ckpt = torch.load(final, map_location='cuda')
model.load_state_dict(ckpt['model'])
model.cuda().eval()
print(f"Loaded model (epoch {ckpt.get('start_epoch', '?')})")

CLASSES = [class_mapping_98[str(i)]["name"] for i in range(NEW_NUM_CLASSES)]
preproc = ValTransform(legacy=False)

# Get images
val_imgs = list(Path(f'{LOCAL_DATA_DIR}/images/val').glob('*.jpg'))[:8]
if len(val_imgs) < 8:
    val_imgs += list(Path(f'{LOCAL_DATA_DIR}/images/val').glob('*.png'))[:8-len(val_imgs)]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, img_path in zip(axes.flat, val_imgs):
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]

    tensor, _ = preproc(img, None, (640, 640))
    tensor = torch.from_numpy(tensor).unsqueeze(0).float().cuda()

    with torch.no_grad():
        out = model(tensor)
        out = postprocess(out, NEW_NUM_CLASSES, 0.25, 0.45)

    n_haz, n_msl = 0, 0
    if out[0] is not None:
        det = out[0].cpu().numpy()
        scale = min(640/h, 640/w)
        for box, score, cls in zip(det[:,:4]/scale, det[:,4]*det[:,5], det[:,6].astype(int)):
            x0,y0,x1,y1 = map(int, box)
            if cls == 97:
                n_msl += 1
                color = (0, 165, 255)  # Orange for MSL
            else:
                n_haz += 1
                color = (0, 255, 0)  # Green for hazmat
            cv2.rectangle(img, (x0,y0), (x1,y1), color, 2)
            cv2.putText(img, f"{CLASSES[cls][:10]}:{score:.2f}", (x0,y0-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.35, color, 1)

    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"Haz:{n_haz} MSL:{n_msl}", fontsize=10)
    ax.axis('off')

plt.suptitle('98-Class Model: Green=Hazmat, Orange=MSL', fontsize=14)
plt.tight_layout()
plt.savefig(f"{DRIVE_OUTPUT}/inference_results.png", dpi=150)
plt.show()

## Cell 12: Export to ExecuTorch

In [ ]:
!pip install executorch -q

from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner

class ExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.m.eval()
        if hasattr(self.m.head, 'decode_in_inference'):
            self.m.head.decode_in_inference = False
    def forward(self, x):
        return self.m(x)

print("Exporting to ExecuTorch...")

model = create_model(NEW_NUM_CLASSES)
ckpt = torch.load(f"{DRIVE_OUTPUT}/msl_98class_final.pth", map_location='cpu')
model.load_state_dict(ckpt['model'])
model.eval()

wrapper = ExportWrapper(model)
exported = export(wrapper, (torch.randn(1, 3, 640, 640),))

try:
    edge = to_edge_transform_and_lower(exported, partitioner=[XnnpackPartitioner()])
except:
    from executorch.exir import to_edge
    edge = to_edge(exported)

et_prog = edge.to_executorch()

pte_path = f"{DRIVE_OUTPUT}/yolox_msl_98class.pte"
with open(pte_path, 'wb') as f:
    f.write(et_prog.buffer)

print(f"\nSaved: {pte_path}")
print(f"Size: {os.path.getsize(pte_path)/1024/1024:.2f} MB")

## Cell 13: Summary

In [ ]:
print("=" * 60)
print("DONE! Single MSL Class Model Ready")
print("=" * 60)

print(f"\nOutput: {DRIVE_OUTPUT}")
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    sz = os.path.getsize(f"{DRIVE_OUTPUT}/{f}") / 1024 / 1024
    print(f"  {f} ({sz:.1f} MB)")

print(f"\n" + "-" * 60)
print("TO UPDATE YOUR APP:")
print("-" * 60)
print(f"""
1. Copy .pte to assets/models/

2. Update src/ml/types/detection.ts:
   numClasses: 98

3. Update src/ml/data/class_mapping.json:
   Add class 97: "militaryShippingLabel"

4. In your app logic:
   hasMSL = detections.some(d => d.classId === 97)

5. Rebuild:
   npx expo run:ios
""")